Pruebo si la sesión de spark crea Data frame

In [2]:
import os
import sys
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"


os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable


from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("test") \
    .getOrCreate()

print("🔥 Spark arrancó correctamente")

df = spark.createDataFrame([{"a": 1}, {"a": 2}])

df.show()   # 👈 ESTO FALTABA

spark.stop()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/14 12:26:52 WARN Utils: Your hostname, DESKTOP-D5DRVDO, resolves to a loopback address: 127.0.1.1; using 172.18.78.22 instead (on interface eth0)
26/05/14 12:26:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/14 12:26:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


🔥 Spark arrancó correctamente


+---+
|  a|
+---+
|  1|
|  2|
+---+



Pruebo test_mfcc

probando con funciones separadas

In [3]:
#codigo de prueba sin trasformar a mfcc
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf
from pyspark.sql.types import ArrayType, FloatType


import librosa

In [4]:
# 🔹 Función que transforma UN audio → MFCC
def extract_mfcc(path):
    try:
        # cargar audio
        y, sr = librosa.load(path, sr=22050)

        # extraer MFCC
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)

        # convertir a lista (Spark no guarda numpy directo)
        return mfcc.flatten().tolist()

    except Exception as e:
        print(f"❌ Error en {path}: {e}")
        return []

In [5]:
def process_mfcc(data):

    # 1. Crear e iniciar sesión de Spark
    spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Audio MFCC Pipeline") \
    .getOrCreate()

    # 👈 ESTA ES LA CLAVE: Envía el archivo al clúster de Spark
    # Esto permite que los workers encuentren el módulo 'etl'
    #spark.sparkContext.addPyFile("etl/mfcc_pyspark.py")

    # 2. Realizar operaciones con la sesión de Spark
    # Convertir lista Python → DataFrame Spark    
    df = spark.createDataFrame(data)

    # 3. Registrar función como UDF (User Defined Function)
    mfcc_udf = udf(extract_mfcc, ArrayType(FloatType()))

    # 4. Aplicar transformación (columna nueva)
    df = df.withColumn("mfcc", mfcc_udf(df["path"]))

    print("✅ MFCC generados con PySpark")
    
    #df.printSchema()
    df.show(5, truncate=False)

    
    return df

In [8]:
def save_data(df, output_path):
    print(f"💾 Guardando en: {output_path}")

    df.write \
        .mode("overwrite") \
        .parquet(output_path)

    print("✅ Guardado completado")

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))
from etl.ingest import ingest_data
#from etl.mfcc_pyspark import process_mfcc

data = ingest_data("../data/raw")
df = process_mfcc(data)

# Definimos la ruta de salida
path_procesado = "data/processed"
# Llamamos a la función pasándole tu DataFrame y la ruta
save_data(df, path_procesado)


df.show(5)


✅ Total audios cargados: 240
✅ MFCC generados con PySpark


+----------+-----+-----------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

✅ Guardado completado
+----------+-----+--------------------+--------------------+
|class_name|label|                path|                mfcc|
+----------+-----+--------------------+--------------------+
|      Asma|    0|../data/raw/Asma/...|[-407.148, -474.1...|
|      Asma|    0|../data/raw/Asma/...|[-423.8708, -454....|
|      Asma|    0|../data/raw/Asma/...|[-160.73915, -228...|
|      Asma|    0|../data/raw/Asma/...|[-502.26944, -498...|
|      Asma|    0|../data/raw/Asma/...|[-494.90118, -506...|
+----------+-----+--------------------+--------------------+
only showing top 5 rows


26/05/14 16:02:41 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 5994677 ms exceeds timeout 120000 ms
26/05/14 16:02:41 WARN SparkContext: Killing executors is not supported by current scheduler.
26/05/14 16:02:42 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:81)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:674)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1363)
	at 